# Poisson 2D en Fortran — portage GPU sur Google Colab

**Avant tout :** menu *Exécution* → *Modifier le type d'exécution* → **GPU T4** → Enregistrer.

Lance ensuite les cellules dans l'ordre (Maj + Entrée). L'installation du compilateur NVIDIA prend 10 à 15 minutes.

In [ ]:
# 1. Vérifier qu'un GPU NVIDIA est bien attribué
!nvidia-smi

## 2. Récupérer le projet
Si le dépôt est déjà sur GitHub, lance la cellule A. Sinon, lance la cellule B et choisis `poisson-2d-fortran-gpu.zip`.

In [ ]:
# A. Depuis GitHub
!git clone https://github.com/abdous-jpg/poisson-2d-fortran-gpu

In [ ]:
# B. Depuis le zip (si le dépôt n'est pas encore sur GitHub)
from google.colab import files
files.upload()
!unzip -oq poisson-2d-fortran-gpu.zip

In [ ]:
%cd /content/poisson-2d-fortran-gpu
!ls

## 3. Installer gfortran et le NVIDIA HPC SDK (compilateur `nvfortran`)

In [ ]:
!apt-get install -y -qq gfortran > /dev/null
!curl -s https://developer.download.nvidia.com/hpc-sdk/ubuntu/DEB-GPG-KEY-NVIDIA-HPC-SDK | gpg --yes --dearmor -o /usr/share/keyrings/nvidia-hpcsdk-archive-keyring.gpg
!echo 'deb [signed-by=/usr/share/keyrings/nvidia-hpcsdk-archive-keyring.gpg] https://developer.download.nvidia.com/hpc-sdk/ubuntu/amd64 /' > /etc/apt/sources.list.d/nvhpc.list
!apt-get update -qq

In [ ]:
# Installe la version la plus récente du SDK (10 à 15 minutes)
import subprocess
paquet = subprocess.run("apt-cache search --names-only '^nvhpc-[0-9]+-[0-9]+$' | cut -d' ' -f1 | sort -V | tail -1",
                        shell=True, capture_output=True, text=True).stdout.strip()
print("Paquet installé :", paquet)
!apt-get install -y -qq $paquet > /dev/null

In [ ]:
import glob, os, re
dossiers = [d for d in glob.glob("/opt/nvidia/hpc_sdk/Linux_x86_64/*/compilers/bin") if re.search(r"/\d+\.\d+/", d)]
bindir = sorted(dossiers, key=lambda d: [int(x) for x in re.search(r"/(\d+)\.(\d+)/", d).groups()])[-1]
os.environ["PATH"] = bindir + ":" + os.environ["PATH"]
!nvfortran --version

## 4. Compiler et vérifier sur GPU
La sortie de `-Minfo=mp` doit indiquer que les boucles sont générées pour le GPU. Le test de convergence doit afficher **OK : ordre 2 retrouvé** : le GPU donne la bonne solution.

In [ ]:
!mkdir -p results && make gpu_test

In [ ]:
!make gpu && make && make simple

In [ ]:
# Les lancements de noyaux GPU s'affichent : preuve que le calcul tourne sur le GPU
!NVCOMPILER_ACC_NOTIFY=1 ./performance_gpu 65 2 2>&1 | head -8

## 5. Mesures CPU vs GPU (500 itérations)

In [ ]:
import subprocess, pandas as pd
versions = [("CPU double, 2 threads", "./performance"), ("CPU simple, 2 threads", "./performance_simple"),
            ("GPU double", "./performance_gpu"), ("GPU simple", "./performance_gpu_simple")]
lignes = []
for nom, prog in versions:
    for n in [513, 1025, 2049, 4097]:
        sortie = subprocess.run(f"OMP_NUM_THREADS=2 {prog} {n} 500", shell=True, capture_output=True, text=True).stdout.strip()
        _, iters, ms, mpts, octets = sortie.split(",")
        lignes.append({"version": nom, "n": n, "ms_par_iteration": float(ms), "mpoints_par_s": float(mpts)})
        print(nom, n, ms.strip(), "ms/it")
res = pd.DataFrame(lignes)
res.to_csv("results/gpu.csv", index=False)
ref = res[res.version == "CPU double, 2 threads"].set_index("n").ms_par_iteration
res["acceleration_vs_cpu_double"] = res.apply(lambda r: ref[r.n] / r.ms_par_iteration, axis=1).round(1)
res.pivot(index="version", columns="n", values="acceleration_vs_cpu_double")

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6.5, 4))
for nom, g in res.groupby("version"):
    ax.plot(g.n, g.mpoints_par_s, "o-", label=nom)
ax.set_xscale("log", base=2); ax.set_yscale("log")
ax.set_xticks([513, 1025, 2049, 4097], ["513", "1025", "2049", "4097"])
ax.set_xlabel("taille de grille n (n x n points)"); ax.set_ylabel("Mpoints / s")
ax.set_title("Jacobi : CPU (2 threads) vs GPU T4"); ax.grid(True, which="both", alpha=0.3); ax.legend()
fig.tight_layout(); fig.savefig("results/gpu.png", dpi=150)

## 6. Récupérer les résultats
Télécharge le zip, puis ajoute `results/gpu.csv` et `results/gpu.png` à ton dépôt, et envoie-moi les chiffres pour mettre à jour le README et ton CV.

In [ ]:
!nvidia-smi --query-gpu=name --format=csv,noheader > results/gpu_modele.txt
!zip -qr resultats_gpu.zip results
from google.colab import files
files.download('resultats_gpu.zip')